In [0]:
# Gold layer: dim_date (Kimball calendar dimension)
# Source: no business source - a generated calendar whose bounds are
#         derived at refresh time from the three silver date columns
#         the gold facts carry: silver.orders.order_date (valid rows
#         only: the 192 null + 214 future dates are excluded by the
#         order_date_valid reporting rule), silver.leads.created_date
#         and silver.logs.event_time (Argentina local time). Verified
#         window: 2024-10-29 .. 2026-11-06 - all three sources share
#         it; the range grows automatically as new silver data
#         arrives, nothing is hardcoded.
# Grain:  exactly one row per calendar day. Smart surrogate key date_sk
#         = YYYYMMDD (e.g. 2024-10-29 -> 20241029) - deterministic, so
#         identical for the same day on every refresh and stable for
#         fact joins.
# Joins:  fact_order_items / fact_leads / fact_web_logs will carry
#         date_sk as the date-dimension foreign key (their date
#         columns are kept raw; invalid/future dates simply do not
#         match a row here, which is exactly the net-revenue
#         reporting rule).
# Columns: only the calendar attributes the 15 KPIs roll up on - KPI
#         1 (monthly revenue trend + MoM growth) and KPI 5 (CLV
#         cohorts) roll up at month grain on calendar_year /
#         calendar_month / year_month, and the facts need date_sk for
#         joins. calendar_quarter, month_name, day_of_month,
#         day_of_week, day_name and is_weekend were dropped in the
#         2026-11-18 KPI audit: no KPI rolls up by quarter,
#         weekday/weekend, day-of-month or month name - the Q4-peak
#         and weekday patterns in requirements.md are
#         data-generation specs, not reports.
# Storage: Delta-backed STREAMING TABLE (user-requested change
#         2026-11-18: was a materialized view, now a stored table),
#         loaded via AUTO CDC FROM SNAPSHOT - the same
#         ingestion-time pattern as the other four gold dims.
# MERGE / idempotency: keyed on date_sk (requirements' MERGE
#         constraint - safe to retry without duplicates). There is no
#         MERGE source table to key on - the calendar is fully derived
#         - so the snapshot diff IS the idempotent load: an identical
#         calendar snapshot diffs to zero events (retries and no-op
#         runs change nothing) and a newly arrived day inserts
#         exactly one row.
# Quality: no expectations - every row is produced by Spark date
#         functions on generated days, so there is no source dirt to
#         catch (silver already owns row-level validation), and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT
#         flows anyway (same decision as the other dims).
# Layout: liquid-clustered by date_sk (dims cluster by their surrogate
#         key - KPI views join on it).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_ORDERS = "b2b_ecommerce.silver.orders"
SILVER_LEADS = "b2b_ecommerce.silver.leads"
SILVER_LOGS = "b2b_ecommerce.silver.logs"


# The complete calendar snapshot for the current event window, trimmed
# to the KPI-required columns. A batch temporary view: the engine reads
# it at each pipeline run (ingestion-time snapshot versioning). No
# driver-side actions anywhere.
@dp.temporary_view(name="date_calendar_snapshot")
def date_calendar_snapshot():
    # 1) Window bounds: min/max day per fact date source, then overall
    #    min / overall max across the union (one row). Each source is
    #    a cheap full-scan aggregate over its own silver table.
    orders_bounds = (
        spark.read.table(SILVER_ORDERS)
        .filter(F.col("order_date_valid"))
        .select(
            F.min(F.col("order_date")).alias("_min_day"),
            F.max(F.col("order_date")).alias("_max_day"),
        )
    )
    leads_bounds = spark.read.table(SILVER_LEADS).select(
        F.min(F.col("created_date")).alias("_min_day"),
        F.max(F.col("created_date")).alias("_max_day"),
    )
    logs_bounds = spark.read.table(SILVER_LOGS).select(
        F.min(F.col("event_time").cast("date")).alias("_min_day"),
        F.max(F.col("event_time").cast("date")).alias("_max_day"),
    )
    bounds = (
        orders_bounds.unionByName(leads_bounds)
        .unionByName(logs_bounds)
        .select(
            F.min(F.col("_min_day")).alias("first_day"),
            F.max(F.col("_max_day")).alias("last_day"),
        )
    )

    # 2) One row per day between the bounds: native sequence + explode
    #    keeps the whole generation in Spark (no driver-side loops).
    calendar = bounds.select(
        F.explode(F.expr("sequence(first_day, last_day, interval 1 day)")).alias(
            "full_date"
        )
    )

    # 3) Calendar attributes, all native expressions on full_date -
    #    the KPI-required set only.
    return calendar.select(
        # smart surrogate key YYYYMMDD (e.g. 2024-10-29 -> 20241029)
        (
            F.year(F.col("full_date")) * 10000
            + F.month(F.col("full_date")) * 100
            + F.dayofmonth(F.col("full_date"))
        )
        .cast("int")
        .alias("date_sk"),
        F.col("full_date").cast("date").alias("full_date"),
        F.year(F.col("full_date")).cast("int").alias("calendar_year"),
        F.month(F.col("full_date")).cast("int").alias("calendar_month"),
        # single sortable month key for monthly trends + MoM growth (202410)
        (F.year(F.col("full_date")) * 100 + F.month(F.col("full_date")))
        .cast("int")
        .alias("year_month"),
    )


# Delta-backed streaming table: the AUTO CDC target, clustered on the
# surrogate key (dims cluster by their surrogate key).
dp.create_streaming_table(
    name="dim_date",
    comment=(
        "Kimball calendar dimension: one row per calendar day covering the "
        "gold fact event window, derived from silver orders.order_date (valid "
        "rows only), leads.created_date and logs.event_time (currently "
        "2024-10-29 to 2026-11-06, grows with new silver data). Smart "
        "surrogate key date_sk = YYYYMMDD is the join key carried by all "
        "three gold facts. Stored as a Delta streaming table MERGE-loaded "
        "via AUTO CDC FROM SNAPSHOT keyed on date_sk - identical snapshots "
        "diff to zero events, so retries are idempotent. Carries only the "
        "month-grain calendar attributes the KPIs roll up on (KPI 1 "
        "monthly revenue trend + MoM growth, KPI 5 CLV cohorts): "
        "calendar_year, calendar_month, year_month - quarter, month-name, "
        "day-of-week and weekend attributes were dropped after the "
        "2026-11-18 KPI audit (no KPI uses them)."
    ),
    cluster_by=["date_sk"],
)


# The auto-merge load: diff each calendar snapshot against the last
# committed one and apply inserts/updates/deletes in place (Type 1 -
# no history columns; a calendar day's attributes never change).
dp.create_auto_cdc_from_snapshot_flow(
    target="dim_date",
    source="date_calendar_snapshot",  # ingestion-time snapshot versioning
    keys=["date_sk"],
    stored_as_scd_type=1,
)